# AI651 – Deep Learning for Space, Time and Graphs
# Assignment 3

Syed Obaidullah Hassan Chishti - 25280049

## Question 1

# Imports

In [1]:
import numpy as np
from itertools import product
import matplotlib.pyplot as plt

# Question 1 — What does a sum remember?

## Part (a) — An affine message cannot count

In [ ]:
def affine_message(x, w, c):
    return w * np.array(x, dtype=float) + c

def aggregate_sum(messages):
    return np.sum(messages, axis=0)

A = [-1, +1, 0, 0]     
B = [-1, -1, +1, +1]  

print("Neighbourhood A:", A, "| n =", len(A), "| S =", sum(A), "| steady =", A.count(0))
print("Neighbourhood B:", B, "| n =", len(B), "| S =", sum(B), "| steady =", B.count(0))

Neighbourhood A: [-1, 1, 0, 0] | n = 4 | S = 0 | steady = 2
Neighbourhood B: [-1, -1, 1, 1] | n = 4 | S = 0 | steady = 0


In [ ]:
rng = np.random.default_rng(0)

print("Aggregated message (sum) for 5 random (w, c):")
for _ in range(5):
    w, c = rng.normal(), rng.normal()
    sa = aggregate_sum(affine_message(A, w, c))
    sb = aggregate_sum(affine_message(B, w, c))
    print(f"w={w:+.3f}, c={c:+.3f}  ->  sum(A)={sa:+.4f}, sum(B)={sb:+.4f}, equal={np.isclose(sa, sb)}")

Aggregated message (sum) for 5 random (w, c):
w=+0.126, c=-0.132  ->  sum(A)=-0.5284, sum(B)=-0.5284, equal=True
w=+0.640, c=+0.105  ->  sum(A)=+0.4196, sum(B)=+0.4196, equal=True
w=-0.536, c=+0.362  ->  sum(A)=+1.4464, sum(B)=+1.4464, equal=True
w=+1.304, c=+0.947  ->  sum(A)=+3.7883, sum(B)=+3.7883, equal=True
w=-0.704, c=-1.265  ->  sum(A)=-5.0617, sum(B)=-5.0617, equal=True


## Part (b) — A nonlinear message that can count

In [ ]:
K = 8
base = K + 1   

def phi_b(x, base=base):
    table = {-1: 1, 0: base, +1: base**2}
    return table[x]

def decode(total, base=base):
    n_slow = total % base
    total //= base
    n_steady = total % base
    total //= base
    n_speed = total
    return n_slow, n_steady, n_speed

print("base =", base)

base = 9


In [ ]:
all_ok = True

for n_slow, n_steady, n_speed in product(range(K + 1), repeat=3):
    size = n_slow + n_steady + n_speed
    if size < 1 or size > K:
        continue

    neighbourhood = [-1]*n_slow + [0]*n_steady + [+1]*n_speed
    total = sum(phi_b(x) for x in neighbourhood)

    if decode(total) != (n_slow, n_steady, n_speed):
        all_ok = False
        print("MISMATCH at", (n_slow, n_steady, n_speed))

print("Decoding correct for every triple with 1 <= |N(v)| <=", K, ":", all_ok)

Decoding correct for every triple with 1 <= |N(v)| <= 8 : True


In [ ]:
ex = [-1, -1, 0, +1, +1, +1]
tot = sum(phi_b(x) for x in ex)

print("Example neighbourhood:", ex)
print("Summed message:", tot)
print("Decoded (slow, steady, speed):", decode(tot))

Example neighbourhood: [-1, -1, 0, 1, 1, 1]
Summed message: 254
Decoded (slow, steady, speed): (2, 1, 3)


In [7]:
def phi_nonsteady(x):
    return {-1: 1, 0: 0, +1: 1}[x]

for nb in [[-1, 0, +1], [0, 0, 0], [-1, -1, +1, 0]]:
    print(nb, "-> non-steady count =", sum(phi_nonsteady(x) for x in nb))

[-1, 0, 1] -> non-steady count = 2
[0, 0, 0] -> non-steady count = 0
[-1, -1, 1, 0] -> non-steady count = 3


## Part (c) — Comparing Sum, Mean and Max with one-hot messages

In [8]:
onehot = {-1: np.array([1, 0, 0]),
           0: np.array([0, 1, 0]),
          +1: np.array([0, 0, 1])}

def messages(nb):
    return np.array([onehot[x] for x in nb], dtype=float)

def agg_sum(nb):  return messages(nb).sum(axis=0)
def agg_mean(nb): return messages(nb).mean(axis=0)
def agg_max(nb):  return messages(nb).max(axis=0)

### Where each method works (the "Yes" cases)

In [ ]:
print("(i) Count steady neighbours  [Sum]")

for nb in [[0, -1, 0, +1], [0, 0, 0]]:
    print("   ", nb, "-> steady count =", int(agg_sum(nb)[1]))

print("\n(ii) More than half slowing")

for nb in [[-1, -1, 0], [-1, 0, 0]]:
    s = agg_sum(nb); n = s.sum()                 
    print("    [Sum] ", nb, "-> slow > half?", bool(s[0] > n / 2))

for nb in [[-1, -1, 0], [-1, 0, 0]]:
    m = agg_mean(nb)                              
    print("    [Mean]", nb, "-> slow > half?", bool(m[0] > 0.5))

print("\n(iii) Any neighbour slowing")

for nb in [[0, +1], [-1, 0]]:
    print("    [Sum] ", nb, "-> any slow?", bool(agg_sum(nb)[0] > 0))
    print("    [Mean]", nb, "-> any slow?", bool(agg_mean(nb)[0] > 0))
    print("    [Max] ", nb, "-> any slow?", bool(agg_max(nb)[0] == 1))

(i) Count steady neighbours  [Sum]
    [0, -1, 0, 1] -> steady count = 2
    [0, 0, 0] -> steady count = 3

(ii) More than half slowing
    [Sum]  [-1, -1, 0] -> slow > half? True
    [Sum]  [-1, 0, 0] -> slow > half? False
    [Mean] [-1, -1, 0] -> slow > half? True
    [Mean] [-1, 0, 0] -> slow > half? False

(iii) Any neighbour slowing
    [Sum]  [0, 1] -> any slow? False
    [Mean] [0, 1] -> any slow? False
    [Max]  [0, 1] -> any slow? False
    [Sum]  [-1, 0] -> any slow? True
    [Mean] [-1, 0] -> any slow? True
    [Max]  [-1, 0] -> any slow? True


In [10]:
print("(i) Count steady neighbours -- Mean and Max cannot count (size is lost):")
print("    Mean: [0] ->", agg_mean([0]), "steady=1   vs  [0,0] ->", agg_mean([0,0]), "steady=2")
print("    Max : [0] ->", agg_max([0]),  "steady=1   vs  [0,0] ->", agg_max([0,0]),  "steady=2")

print("\n(ii) More than half slowing -- Max cannot test a majority:")
print("    Max: [-1,0] ->", agg_max([-1,0]), "(1 of 2 slow -> No)")
print("    Max: [-1,-1,0] ->", agg_max([-1,-1,0]), "(2 of 3 slow -> Yes)")
print("    Same vector (1,1,0) but different answers.")

(i) Count steady neighbours -- Mean and Max cannot count (size is lost):
    Mean: [0] -> [0. 1. 0.] steady=1   vs  [0,0] -> [0. 1. 0.] steady=2
    Max : [0] -> [0. 1. 0.] steady=1   vs  [0,0] -> [0. 1. 0.] steady=2

(ii) More than half slowing -- Max cannot test a majority:
    Max: [-1,0] -> [1. 1. 0.] (1 of 2 slow -> No)
    Max: [-1,-1,0] -> [1. 1. 0.] (2 of 3 slow -> Yes)
    Same vector (1,1,0) but different answers.
